In [92]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import warnings

from pathlib import Path
import re
import shutil

from datetime import datetime

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [93]:
LATEST_PARTITION = "2026-07-06"

In [94]:
def read_sheets(filename):
    """Read the input data"""
    sheets = pd.read_excel(filename, sheet_name=None)
    df = pd.concat(sheets.values(), ignore_index=True)
    return df.shape, df

df_time_shape, df_time = read_sheets("data/IQ Time card data - 06072026.xls")
df_time = df_time[~df_time["Rate type"].str.contains("On-Call", na=False)]
print(f"Time shape: {df_time_shape}")

Time shape: (1375, 16)


In [95]:
df_time = df_time.drop_duplicates(keep='last')
df_time.shape

(1375, 16)

In [96]:
TECH_MAP = {
    "Dynamics 365 FO": "F&O",
    "Dynamics 365 FSCM": "F&O",
    "Dynamics 365 CE": "CE",
    "Dynamics 365 BC": "BC",
    "Dynamics NAV": "BC",
    "Microsoft Azure": "Azure Integ",
    "Microsoft Cloud Data Warehouse": "Inf Mgmt",
    "Microsoft Cloud Support Infrastructure": "Inf Mgmt",
    "Power Platform": "Power Platform",
}

df_time["tech_group"] = df_time["Technology"].map(TECH_MAP).fillna("Other")

In [97]:
df_time["tech_group"].value_counts(dropna=False)

tech_group
F&O               789
CE                255
Other             210
Inf Mgmt           36
Power Platform     31
BC                 29
Azure Integ        25
Name: count, dtype: int64

In [98]:
df_time.shape

(1375, 17)

In [99]:
df_time["Date"].max()

Timestamp('2026-07-11 00:00:00')

In [100]:
df_recent = df_time[df_time["Date"]>=LATEST_PARTITION]

In [101]:
df_recent.shape

(1375, 17)

## Get FTEs without GEN or leaves

In [102]:
df_recent = df_recent[
    (df_recent["Category"] == "Task work") &
    (~df_recent["Task"].str[:3].eq("GEN"))
]

In [103]:
df_recent.shape

(817, 17)

In [104]:
df_recent = df_recent.drop(
    ["Task", "Category", "User ID", "Rate type",
     "Additional comments", "Grade", "Business Services",
     "Short description", "Company", "Business service", "Project ID", "User"], axis=1)

In [106]:
TECH_MAP = {
    "Dynamics 365 FO": "F&O",
    "Dynamics 365 FSCM": "F&O",
    "Dynamics 365 CE": "CE",
    "Dynamics 365 BC": "BC",
    "Dynamics NAV": "BC",
    "Microsoft Azure": "Azure Integ",
    "Microsoft Cloud Data Warehouse": "Inf Mgmt",
    "Microsoft Cloud Support Infrastructure": "Inf Mgmt",
    "Power Platform": "Power Platform",
}

df_recent["tech_group"] = df_recent["Technology"].map(TECH_MAP).fillna("Other")

In [107]:
df_recent["tech_group"].value_counts(dropna=False)

tech_group
F&O               515
CE                199
Inf Mgmt           36
Azure Integ        25
BC                 24
Power Platform     15
Other               3
Name: count, dtype: int64

In [108]:
df_recent.drop(["Technology", "Specialisation"], axis=1, inplace=True)

In [109]:
summary = (
    df_recent
    .groupby('tech_group', as_index=False)
    .agg(Total_Time_Worked=('Time worked', 'sum'))
)

summary['FTE_85pct'] = summary['Total_Time_Worked'] / (40*0.85)


total_row = pd.DataFrame({
    'tech_group': ['TOTAL'],
    'Total_Time_Worked': [summary['Total_Time_Worked'].sum()],
    'FTE_85pct': [summary['FTE_85pct'].sum()]
})

summary = pd.concat([summary, total_row], ignore_index=True)

display(summary.sort_values('Total_Time_Worked', ascending=False))

,tech_group,Total_Time_Worked,FTE_85pct
7,TOTAL,1391.03,40.912647
3,F&O,824.40,24.247059
2,CE,344.95,10.145588
4,Inf Mgmt,71.75,2.110294
0,Azure Integ,53.25,1.566176
6,Power Platform,47.00,1.382353
1,BC,43.68,1.284706
5,Other,6.00,0.176471


## Get FTE considering GEN and leaves but no on-call

In [110]:
df_recent_all = df_time[df_time["Date"]>=LATEST_PARTITION]

In [111]:
df_recent_all = df_recent_all[~df_recent_all["Rate type"].str.contains("On-Call", na=False)]
df_recent_all.shape

(1375, 17)

In [112]:
df_recent_all["tech_group"] = df_recent_all["Technology"].map(TECH_MAP)
df_recent_all.drop(["Technology", "Specialisation", "Task", "Category", "User ID", "Rate type",
     "Additional comments", "Grade", "Business Services",
     "Short description", "Company", "Business service", "Project ID", "User"], axis=1, inplace=True)

In [113]:
total_time_worked = df_recent_all['Time worked'].sum()
fte_85pct = total_time_worked / (40 * 0.85)

summary = pd.DataFrame({
    'Total_Time_Worked': [total_time_worked],
    'FTE_85pct': [fte_85pct]
})

display(summary)

,Total_Time_Worked,FTE_85pct
0,2721.18,80.034706
